In [0]:
# CELL 2
from pyspark.sql.functions import *
from delta.tables import DeltaTable
import logging

logger = logging.getLogger("gold_orders")
logging.basicConfig(level=logging.INFO)

In [0]:
df = spark.sql("select * from medillian_upsert.silver.orders")
df.display()

In [0]:
%sql
select * from medillian_upsert.gold.dimproducts

In [0]:
%sql
select * from medillian_upsert.gold.dimcustomers

In [0]:

df = spark.read.table("medillian_upsert.silver.orders")
df_dimcus = spark.sql("SELECT DimCustomerKey, customer_id AS dim_customer_id FROM medillian_upsert.gold.DimCustomers")
df_dimpro = spark.sql("SELECT DimProductsKey, product_id AS dim_product_id FROM medillian_upsert.gold.DimProducts")

input_count = df.count()
logger.info(f"Input rows from silver.orders: {input_count:,}")


In [0]:
# CELL 4 — Data quality checks BEFORE join
dq_issues = {}

null_order_ids = df.filter(col("order_id").isNull()).count()
null_customers = df.filter(col("customer_id").isNull()).count()
null_products  = df.filter(col("product_id").isNull()).count()
negative_qty   = df.filter(col("quantity") <= 0).count()
negative_amt   = df.filter(col("total_amount") <= 0).count()

if null_order_ids > 0:  dq_issues["null_order_ids"]  = null_order_ids
if null_customers > 0:  dq_issues["null_customers"]  = null_customers
if null_products  > 0:  dq_issues["null_products"]   = null_products
if negative_qty   > 0:  dq_issues["negative_qty"]    = negative_qty
if negative_amt   > 0:  dq_issues["negative_amt"]    = negative_amt

if dq_issues:
    logger.warning(f"DQ issues found: {dq_issues}")
    # Drop bad rows — don't let them poison the fact table
    df = df.filter(
        col("order_id").isNotNull() &
        col("customer_id").isNotNull() &
        col("product_id").isNotNull() &
        (col("quantity") > 0) &
        (col("total_amount") > 0)
    )
    logger.info(f"Rows after DQ filter: {df.count():,}")
else:
    logger.info("DQ checks passed — no issues found.")

In [0]:
# CELL 5 — Join + check for nulls (unresolved foreign keys)
df_fact = df.join(df_dimcus, df["customer_id"] == df_dimcus["dim_customer_id"], "left") \
            .join(df_dimpro, df["product_id"]  == df_dimpro["dim_product_id"],  "left")


In [0]:
df_fact.display()

In [0]:
# Alert if any foreign keys didn't resolve — means dim tables are out of sync
unresolved_customers = df_fact.filter(col("DimCustomerKey").isNull()).count()
unresolved_products  = df_fact.filter(col("DimProductsKey").isNull()).count()

if unresolved_customers > 0:
    logger.error(f"ALERT: {unresolved_customers:,} orders have no matching DimCustomerKey — dim may be stale!")
if unresolved_products > 0:
    logger.error(f"ALERT: {unresolved_products:,} orders have no matching DimProductsKey — dim may be stale!")

# Hard stop if more than 5% of rows are unresolved — something is seriously wrong
unresolved_pct = (unresolved_customers + unresolved_products) / (input_count * 2)
if unresolved_pct > 0.05:
    raise ValueError(f"Too many unresolved foreign keys ({unresolved_pct:.1%}) — aborting to protect fact table.")

In [0]:
# CELL 6 — Clean + deduplicate
df_fact = df_fact.drop("dim_customer_id", "dim_product_id", "customer_id", "product_id") \
                 .dropDuplicates(["order_id"])

logger.info(f"Rows going into MERGE: {df_fact.count():,}")

In [0]:
df_fact.display()

In [0]:
from delta.tables import DeltaTable

In [0]:
# Upsert on fact
if spark.catalog.tableExists("medillian_upsert.gold.factOrders"):

    dlt_obj = DeltaTable.forName(spark, "medillian_upsert.gold.factOrders")

    # Perform the merge on natural business key only
    dlt_obj.alias("trg").merge(
        df_fact.alias("src"),
        "trg.order_id = src.order_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

else:
    df_fact.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("medillian_upsert.gold.factOrders")

In [0]:
%sql
select * from medillian_upsert.gold.factorders

In [0]:
%sql
select count(order_id) cnt from medillian_upsert.gold.factorders

In [0]:
%sql
select order_id , count(order_id) cnt from medillian_upsert.gold.factorders group by order_id  having cnt>1 order by cnt desc;